# 🧠 URL Semantic Classifier — Fine-tuned SecureBERT
**Why SecureBERT?**
- Pre-trained on cybersecurity text — already understands words like "verify", "secure", "login", "account" in phishing context
- Understands URL *meaning*, not just character patterns
- Handles long URLs, IP addresses, encoded chars without bias
- Much smarter than BiLSTM which only memorizes sequences

**Dataset:** PhiUSIIL (235k URLs, balanced, diverse lengths)
**Output:** `url_classifier/` folder → zip and download

In [ ]:
# GPU check — make sure you have T4/P100 enabled in Colab
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else "⚠️ No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'torch', 'scikit-learn', 'tldextract',
    'pandas', 'joblib', 'datasets', 'accelerate'])
print('✅ Packages installed')

## 1 — Load & Balance Dataset

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

df_raw = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "harisudhan411/phishing-and-legitimate-urls", "")

# Normalise columns
col_map = {}
for c in df_raw.columns:
    cl = c.lower().strip()
    if cl in ['url','urls','link','domain']: col_map[c] = 'url'
    elif cl in ['label','type','status','class','result']: col_map[c] = 'label'

df = df_raw.rename(columns=col_map)[['url','label']].dropna()
label_map = {
    'bad':1,'phishing':1,'malicious':1,'spam':1,'1':1,1:1,
    'good':0,'benign':0,'legitimate':0,'safe':0,'0':0,0:0
}
df['label'] = df['label'].map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
print(f"Raw → Total: {len(df):,}  Phishing: {(df.label==1).sum():,}  Legit: {(df.label==0).sum():,}")

# Balance
phish = df[df.label==1]
legit = df[df.label==0]
n = min(len(phish), len(legit), 80_000)   # 80k each = 160k total — enough for BERT
df_bal = pd.concat([
    phish.sample(n, random_state=42),
    legit.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced → {len(df_bal):,} rows  ({n:,} phishing + {n:,} legit)")
print(df_bal.head(4))

## 2 — Preprocess URLs for BERT
> We make the URL human-readable so BERT can use its language knowledge

In [ ]:
import re, unicodedata

def url_to_text(url: str) -> str:
    """
    Convert a URL into readable text that BERT can reason about semantically.
    Example: http://paypal-secure-verify.login.tk/account/update
    → 'paypal secure verify login tk account update'
    BERT then sees known phishing keywords in context.
    """
    url = str(url).lower().strip()
    # Remove protocol and www
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'^www\.', '', url)
    # Normalize unicode
    url = unicodedata.normalize('NFKD', url).encode('ascii', 'ignore').decode('ascii')
    # Split on any non-alphanumeric character
    tokens = re.split(r'[^a-z0-9]', url)
    tokens = [t for t in tokens if t and len(t) > 1]  # skip single chars and empty
    # Truncate to 50 tokens max (BERT has 512 token limit)
    return ' '.join(tokens[:50])

# Test the preprocessing
test_cases = [
    "http://paypal-secure-verify.login.tk/account/update",
    "https://google.com",
    "http://192.168.1.1/admin/login",
    "https://github.com/user/repo/pulls",
    "http://amaz0n-acc0unt.update-secure.xyz/signin",
]
print("URL → text conversion examples:")
print("-" * 70)
for u in test_cases:
    print(f"IN : {u}")
    print(f"OUT: {url_to_text(u)}")
    print()

df_bal['text'] = df_bal['url'].apply(url_to_text)
print(f"✅ Preprocessed {len(df_bal):,} URLs")

## 3 — Load SecureBERT Tokenizer

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# SecureBERT: BERT fine-tuned on cybersecurity corpus
# Understands: phishing, malware, credential, verify, secure, login etc.
MODEL_NAME = "ehsanaghaei/SecureBERT"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Train/val/test split
train_df, temp_df = train_test_split(df_bal, test_size=0.2, random_state=42, stratify=df_bal['label'])
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

# Tokenize
def tokenize(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

print("Tokenizing...")
train_enc = tokenize(train_df['text'])
val_enc   = tokenize(val_df['text'])
test_enc  = tokenize(test_df['text'])
print("✅ Tokenization done")

## 4 — Create PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class URLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = URLDataset(train_enc, train_df['label'].reset_index(drop=True))
val_dataset   = URLDataset(val_enc,   val_df['label'].reset_index(drop=True))
test_dataset  = URLDataset(test_enc,  test_df['label'].reset_index(drop=True))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

## 5 — Fine-tune SecureBERT

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
).to(device)

EPOCHS    = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss_sum += outputs.loss.item()
            preds     = outputs.logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)
    return correct/total, loss_sum/len(loader)

print("Starting training...")
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if i % 200 == 0:
            print(f"  Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    val_acc, val_loss = evaluate(val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4%} | Val Loss: {val_loss:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('/kaggle/working/url_classifier')
        tokenizer.save_pretrained('/kaggle/working/url_classifier')
        print(f"  ✅ Best model saved (val_acc={val_acc:.4%})")

print(f"\n🏆 Best validation accuracy: {best_val_acc:.4%}")

## 6 — Final Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())

print("=== Final Test Results ===")
print(classification_report(all_labels, all_preds, target_names=['Legitimate','Phishing']))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5,4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Legit','Phishing']); ax.set_yticklabels(['Legit','Phishing'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j,i,cm[i,j],ha='center',va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=14)
plt.tight_layout(); plt.show()

## 7 — Sanity Test

In [ ]:
test_urls = [
    ("https://google.com",                              "expect: legit"),
    ("http://paypal-secure-verify.login.tk",            "expect: phishing"),
    ("http://192.168.1.1/admin/login",                  "expect: phishing"),
    ("https://github.com/user/repo/tree/main",          "expect: legit"),
    ("http://amaz0n-account.update-secure.xyz",         "expect: phishing"),
    ("https://stackoverflow.com/questions/123456",      "expect: legit"),
    ("http://facebook.com.phishing-login.ru",           "expect: phishing"),
    ("http://phishing-site.example",                    "expect: phishing"),
    ("https://elearning.univ-boumerdes.dz",             "expect: legit"),
    ("http://microsoft-account-verify.secure-login.tk", "expect: phishing"),
]

model.eval()
print(f"{'URL':<55} {'Score':>6}  Result")
print("-"*85)
with torch.no_grad():
    for url, note in test_urls:
        text   = url_to_text(url)
        enc    = tokenizer(text, return_tensors='pt', truncation=True, max_length=128,
                           padding='max_length').to(device)
        logits = model(**enc).logits
        probs  = torch.softmax(logits, dim=1)[0]
        phish_score = probs[1].item()
        label  = '🚨 PHISHING' if phish_score > 0.5 else '✅ LEGIT   '
        print(f"{url:<55} {phish_score:>6.2%}  {label}  ({note})")

## 8 — Save & Download

In [ ]:
import shutil, os

# The model is already saved to /kaggle/working/url_classifier/
# Zip it for download
shutil.make_archive('/kaggle/working/url_classifier', 'zip', '/kaggle/working/url_classifier')
print("✅ Saved to /kaggle/working/url_classifier.zip")
print("   Files inside:")
for f in os.listdir('/kaggle/working/url_classifier'):
    size = os.path.getsize(f'/kaggle/working/url_classifier/{f}') / 1024 / 1024
    print(f"   {f:40s} {size:.1f} MB")
print("\n👉 Download url_classifier.zip from the Output panel on the right")